# Phát hiện biển số - A1 (YOLO26n vs YOLOv8n)
Thin driver (không phụ thuộc OS). Toàn bộ logic nằm trong `plate_detect`; notebook chỉ gọi CLI.

- **Local kernel** có GPU: repo và `data/` đã sẵn trên đĩa, cell Setup chỉ `pip install` package.
- **Colab runtime** (kể cả qua VSCode, headless): cell Setup clone repo private (GitHub PAT nhập qua prompt), cài CLI, và tải raw A1 từ `A1.zip` chia sẻ trên Drive qua `gdown` (đặt `DRIVE_FILE_ID`). Các cell sau chạy từ gốc repo.

> **Lưu ý headless:** `drive.mount()` và Colab Secrets yêu cầu frontend **web** của Colab và sẽ lỗi khi runtime điều khiển qua VSCode (400 Bad Request / cancelled auth). Do đó PAT nhập qua prompt `getpass` và dữ liệu tải qua `gdown` thay vì Drive mount.

In [1]:
# === Setup ===
# Local kernel: repo đã có sẵn trên đĩa + data đã chuẩn bị -> chỉ cần cài package:
#     %cd /path/to/UIT2026-DoAnCuoiKi
#     !pip install -e src/ml/plate_detection_pipeline
# Colab runtime (kể cả chạy qua VSCode, HEADLESS): clone BRANCH repo private, cài CLI,
#   kéo raw A1 từ zip chia sẻ trên Drive qua gdown.
#   LƯU Ý: drive.mount() và google.colab Secrets KHÔNG hoạt động từ VSCode - cần frontend
#   web của Colab. Nên: PAT qua prompt getpass, data qua gdown (không mount, không secrets).
import os, glob, shutil, getpass

IN_COLAB = "google.colab" in str(get_ipython())
REPO     = "/content/UIT2026-DoAnCuoiKi"
BRANCH   = "feat/plate-detect-a1"              # package CHƯA merge vào main - clone branch này
RAW      = "data/raw/kaggle_vn_plate_segment"  # layout mà A1Adapter mong đợi: {images,labels}/{train,val}
# MyDrive/UIT_2025/datasets/A1.zip, chia sẻ "Anyone with the link" (id lấy từ URL chia sẻ):
DRIVE_FILE_ID = "1hwIns2lhAgg3i9gKdSccVuMil-AmZZty"

if IN_COLAB:
    # 1) clone repo PRIVATE, feature branch. PAT qua prompt (ô nhập hiện trong VSCode/Colab).
    if not os.path.isdir(REPO):
        tok = getpass.getpass("GitHub PAT: ")
        !git clone --branch {BRANCH} --single-branch https://{tok}@github.com/UIT-DoAnCuoiKi/UIT2026-DoAnCuoiKi.git {REPO}
        del tok
    assert os.path.isdir(REPO), "clone failed - PAT lacks read access to the org repo (see README 'PAT setup')"
    %cd {REPO}
    !git rev-parse --abbrev-ref HEAD   # xác nhận đang ở feature branch

    # 2) cài package -> đưa CLI `plate_detect` vào PATH
    !pip install -q -e src/ml/plate_detection_pipeline

    # 3) raw A1 từ zip Drive qua gdown (headless; KHÔNG drive.mount). Symlink RAW tới nó.
    if not os.path.isdir(f"{RAW}/images/train"):
        assert DRIVE_FILE_ID, "set DRIVE_FILE_ID above (share A1.zip 'Anyone with link', copy id from URL)"
        !pip install -q gdown
        !gdown "https://drive.google.com/uc?id={DRIVE_FILE_ID}" -O /tmp/A1.zip
        !unzip -q -o /tmp/A1.zip -d /tmp/a1
        hits = glob.glob("/tmp/a1/**/images/train", recursive=True)
        assert hits, "images/train not found after unzip - inspect /tmp/a1 and adjust"
        root = os.path.abspath(hits[0][: -len("/images/train")])
        os.makedirs(os.path.dirname(RAW), exist_ok=True)
        if os.path.islink(RAW) or os.path.exists(RAW):
            (os.unlink if os.path.islink(RAW) else shutil.rmtree)(RAW)
        os.symlink(root, os.path.abspath(RAW))
else:
    # local kernel: giả định cwd là gốc repo và data/ đã có sẵn
    !pip install -q -e src/ml/plate_detection_pipeline

# 4) kiểm tra layout raw mà adapter đọc (train + val, images + labels)
for s in ("train", "val"):
    for k in ("images", "labels"):
        assert os.path.isdir(f"{RAW}/{k}/{s}"), f"missing {RAW}/{k}/{s} - check zip split names (val vs valid)"
print("OK - CLI installed, raw A1 ready at", RAW)

Cloning into '/content/UIT2026-DoAnCuoiKi'...
remote: Enumerating objects: 961, done.
remote: Counting objects: 100% (383/383), done.
remote: Compressing objects: 100% (212/212), done.
remote: Total 961 (delta 227), reused 277 (delta 160), pack-reused 578 (from 1)
Receiving objects: 100% (961/961), 37.80 MiB | 42.63 MiB/s, done.
Resolving deltas: 100% (519/519), done.
/content/UIT2026-DoAnCuoiKi
feat/plate-detect-a1
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 95.6 MB/s eta 0:00:00:00:0100:01
  Building editable for plate_detect (pyproject.toml) ... done
Downloading...
From (original): https://drive.google.com/uc?id=1hwIns2lhAgg3i9gKdSccVuMil-AmZZty
From (redirected): https://drive.goo

In [2]:
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

CUDA: True Tesla T4


## 1. Chuẩn bị dữ liệu: kiểm tra class-map, chia tập, khử trùng lặp (train/test, train/val) và xác thực

In [3]:
!plate_detect prepare

prepared: {'counts': {'train': 3433, 'val': 573, 'test': 572}, 'dup_train_test': 230, 'dup_train_val': 226, 'class_map': {1: 'bien_2hang', 0: 'bien_1hang'}, 'phash_report': 'data/processed/a1_det/phash_report.txt'}


In [4]:
!plate_detect check

data-contract OK


## 2. Huấn luyện: ma trận đầy đủ @640 (hai model, seed 0,1,2)

In [ ]:
# CHẠY ĐẦY ĐỦ - 100 epochs, cả hai model x seed 0,1,2 (từ configs/default.yaml). Early-stop qua patience=20.
!plate_detect train --config src/ml/plate_detection_pipeline/configs/default.yaml --project runs

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
New https://pypi.org/project/ultralytics/8.4.118 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.37 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=src/ml/plate_detect/configs/a1_det.yaml, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=70, erasing=0.4, exist_ok=True, fliplr=0

## 4. Export mô hình tốt nhất sang ONNX (hai model @640, kiểm parity)
Export **cả hai** yolov8n và yolo26n (best seed mỗi model) để benchmark trên máy đích. Điểm val không đáng tin do A1 rò rỉ near-dup, nên thiết bị chấm lại trên tập test held-out.

In [ ]:
# ví dụ; lặp lại cho best run của mỗi model/imgsz:
!plate_detect export --weights runs/yolo26n_s0_640/weights/best.pt --out weights/yolo26n_a1_640.onnx --imgsz 640

## 5. Đánh giá trên tập test A1: bảng so sánh và experiments.csv

In [ ]:
!plate_detect eval --imgszs 640 --project runs --weights-dir weights --sample-image data/processed/a1_det/images/test/$(ls data/processed/a1_det/images/test | head -1)

## 6. Đóng gói kết quả: lưu lên Drive hoặc tải qua trình duyệt

Nén `runs/`, `weights/`, và `experiments.csv` thành một archive. Ưu tiên mount **Drive** (vẫn còn sau khi runtime hết hạn, không giới hạn dung lượng); dự phòng **tải qua trình duyệt** khi không mount được Drive (VSCode headless, xem lưu ý ở Setup) hoặc khi chạy local kernel.

In [ ]:
# === Đóng gói toàn bộ kết quả vào một .zip, rồi copy lên Drive hoặc tải qua trình duyệt ===
import os, glob, datetime, shutil

STAMP   = datetime.datetime.now().strftime("%Y%m%d_%H%M")
ARCHIVE = f"/content/plate_det_results_{STAMP}.zip" if IN_COLAB else f"plate_det_results_{STAMP}.zip"

# gom những gì tồn tại (runs/ = weights+plots+curves, weights/ = ONNX đã export, experiments.csv)
targets = [p for p in ("runs", "weights", "experiments.csv") if os.path.exists(p)]
assert targets, "nothing to zip - run train/export/eval first"
print("zipping:", targets)
!zip -rq "{ARCHIVE}" {" ".join(targets)}
print("archive:", ARCHIVE, f"({os.path.getsize(ARCHIVE)/1e6:.1f} MB)")

# 1) ưu tiên: copy vào Drive (sống sót qua timeout runtime). mount lỗi trên VSCode-headless.
saved = False
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        dest = "/content/drive/MyDrive/UIT_2025/results"
        os.makedirs(dest, exist_ok=True)
        shutil.copy(ARCHIVE, dest)
        print("copied to Drive:", os.path.join(dest, os.path.basename(ARCHIVE)))
        saved = True
    except Exception as e:
        print("Drive mount unavailable (VSCode-headless?), falling back to download:", repr(e))

# 2) dự phòng: tải qua trình duyệt (Colab web) - với local kernel file đã có sẵn trên đĩa.
if not saved:
    if IN_COLAB:
        from google.colab import files
        files.download(ARCHIVE)
    else:
        print("saved locally at:", os.path.abspath(ARCHIVE))